In [ ]:
import pandas as pd
import joblib
import os

# List all relevant degree_cpc joblib files (adjust paths if needed)
files = [
    'RCSP002_CA_degree_cpc_AddHealth.joblib',
    'RCSP002_CA_degree_cpc_Banerjee.joblib',
    'RS002_CA_degree_cpc_AddHealth.joblib',
    'RS002_CA_degree_cpc_Banerjee.joblib',
]

# Load and concatenate all files, adding 'mode' and 'graph_type' columns
dfs = []
for file_path in files:
    if os.path.exists(file_path):
        try:
            # Use joblib.load (correct for joblib.dump files, which often use compression)
            temp_df = joblib.load(file_path)
            
            # Determine mode (RCS or RS) and graph type from filename
            filename = os.path.basename(file_path)
            if 'RCSP' in filename:
                mode = 'RCS'
            else:
                mode = 'RS'
            if 'AddHealth' in filename:
                graph_type = 'AddHealth'
            elif 'Banerjee' in filename:
                graph_type = 'Banerjee'
            else:
                continue
            
            temp_df['mode'] = mode
            temp_df['graph_type'] = graph_type
            dfs.append(temp_df)
            print(f"Loaded: {file_path} (shape: {temp_df.shape})")
        except Exception as e:
            print(f"Failed to load {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

# Concatenate all dataframes if any were loaded
if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"\nCombined DataFrame shape: {df_all.shape}")
    print("Columns:", df_all.columns.tolist())
else:
    print("No files were successfully loaded. Please check file paths and existence.")

In [ ]:
df_all.head()

In [ ]:
#get a subsample of df_all for faster testing
df = df_all.sample(frac=1, random_state=42)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from joblib import load
from scipy.stats import pearsonr
import numpy as np

# Get unique threshold values (T) and sort them
T_values = sorted(df['T'].unique())

# Define the centralities, including degree
centralities = ['degree', 'betweenness', 'closeness', 'eigenvector']

# Compute correlations for each T and centrality
corrs = []
for T in T_values:
    df_T = df[df['T'] == T].dropna(subset=['CPC'] + centralities)
    for cent in centralities:
        if len(df_T) > 1 and np.var(df_T['CPC']) > 0 and np.var(df_T[cent]) > 0:
            corr, _ = pearsonr(df_T['CPC'], df_T[cent])
        else:
            corr = np.nan
        corrs.append({'T': T, 'centrality': cent, 'correlation': corr})

# Create a DataFrame for plotting
df_corrs = pd.DataFrame(corrs)

# Split data into two groups
df_low = df_corrs[df_corrs['T'] < 1]
df_high = df_corrs[df_corrs['T'] >= 1]

# Get unique T values for each subplot
unique_T_low = sorted(df_low['T'].unique())
unique_T_high = sorted(df_high['T'].unique())

# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Plot for thresholds < 1
sns.lineplot(data=df_low, x='T', y='correlation', hue='centrality', marker='o', ax=ax1)
ax1.set_xlabel(r'$\theta$', fontsize=14)
ax1.set_ylabel('Pearson Correlation')
ax1.set_title('Correlations for Relative Thresholds', fontsize=14)
ax1.set_xticks(unique_T_low)
ax1.set_ylim(0, 1)
ax1.grid(False)
ax1.legend(title='centrality', loc='upper right')

# Plot for thresholds >= 1
sns.lineplot(data=df_high, x='T', y='correlation', hue='centrality', marker='o', ax=ax2)
ax2.set_xlabel('T', fontsize=14)
ax2.set_ylabel('Pearson Correlation between Node Importance and Centrality')
ax2.set_title('Correlations for Absolute Thresholds', fontsize=14)
ax2.set_xticks(unique_T_high)
ax2.set_ylim(0, 1)
ax2.grid(False)
ax2.legend(title='centrality', loc='upper right')

plt.suptitle('Pearson Correlations between Node Importance and Centralities', fontsize=16)
plt.tight_layout()
plt.savefig('correlations_centralities_vs_Node_Importance.png', dpi=300, bbox_inches='tight')